In [1]:
%%configure -f
{
  "defaultLakehouse": {
    "name": "lh_insurance_bronze_silver",
    "id": "c5d42bb8-2bbc-4172-a40b-8fbcdee30e29",
    "workspaceId": "3533e4eb-ba19-4350-b072-e6497106fb81"
  }
}

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, -1, Finished, Available, Finished, True)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 3, Finished, Available, Finished, False)

In [3]:
hub_claim = spark.table("hub_claim")
hub_policy = spark.table("hub_policy")
hub_customer = spark.table("hub_customer")
hub_vehicle = spark.table("hub_vehicle")
hub_payment = spark.table("hub_payment")

link_claim_policy = spark.table("link_claim_policy")
link_policy_customer = spark.table("link_policy_customer")
link_claim_vehicle = spark.table("link_claim_vehicle")
link_claim_payment = spark.table("link_claim_payment")

sat_claim_details = spark.table("sat_claim_details")
sat_policy_coverage = spark.table("sat_policy_coverage")
sat_customer_profile = spark.table("sat_customer_profile")
sat_vehicle_attributes = spark.table("sat_vehicle_attributes")
sat_payment_details = spark.table("sat_payment_details")

silver_claims = spark.table("silver_stg_claims")

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 4, Finished, Available, Finished, False)

In [4]:
w_claim = Window.partitionBy("claim_hk").orderBy(F.col("effective_from").desc(), F.col("load_datetime").desc())
w_policy = Window.partitionBy("policy_hk").orderBy(F.col("effective_from").desc(), F.col("load_datetime").desc())
w_customer = Window.partitionBy("customer_hk").orderBy(F.col("effective_from").desc(), F.col("load_datetime").desc())
w_vehicle = Window.partitionBy("vehicle_hk").orderBy(F.col("effective_from").desc(), F.col("load_datetime").desc())
w_payment = Window.partitionBy("payment_hk").orderBy(F.col("effective_from").desc(), F.col("load_datetime").desc())

latest_claim_sat = sat_claim_details.withColumn("rn", F.row_number().over(w_claim)).filter("rn = 1").drop("rn")
latest_policy_sat = sat_policy_coverage.withColumn("rn", F.row_number().over(w_policy)).filter("rn = 1").drop("rn")
latest_customer_sat = sat_customer_profile.withColumn("rn", F.row_number().over(w_customer)).filter("rn = 1").drop("rn")
latest_vehicle_sat = sat_vehicle_attributes.withColumn("rn", F.row_number().over(w_vehicle)).filter("rn = 1").drop("rn")
latest_payment_sat = sat_payment_details.withColumn("rn", F.row_number().over(w_payment)).filter("rn = 1").drop("rn")

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 5, Finished, Available, Finished, False)

In [5]:
payment_agg = (
    link_claim_payment.alias("lcp")
    .join(hub_payment.alias("hp"), F.col("lcp.payment_hk") == F.col("hp.payment_hk"), "left")
    .join(latest_payment_sat.alias("ps"), F.col("lcp.payment_hk") == F.col("ps.payment_hk"), "left")
    .groupBy(F.col("lcp.claim_hk"))
    .agg(
        F.countDistinct(F.col("hp.payment_number")).alias("payment_count"),
        F.sum(F.col("ps.payment_amount")).alias("total_payment_amount"),
        F.max(F.col("ps.payment_date")).alias("latest_payment_date")
    )
)

dm_claim_360 = (
    hub_claim.alias("hc")
    .join(latest_claim_sat.alias("cs"), "claim_hk", "left")
    .join(link_claim_policy.alias("lcpol"), "claim_hk", "left")
    .join(hub_policy.alias("hp"), F.col("lcpol.policy_hk") == F.col("hp.policy_hk"), "left")
    .join(latest_policy_sat.alias("ps"), F.col("lcpol.policy_hk") == F.col("ps.policy_hk"), "left")
    .join(link_claim_vehicle.alias("lcv"), "claim_hk", "left")
    .join(hub_vehicle.alias("hv"), F.col("lcv.vehicle_hk") == F.col("hv.vehicle_hk"), "left")
    .join(latest_vehicle_sat.alias("vs"), F.col("lcv.vehicle_hk") == F.col("vs.vehicle_hk"), "left")
    .join(link_policy_customer.alias("lpc"), F.col("lcpol.policy_hk") == F.col("lpc.policy_hk"), "left")
    .join(hub_customer.alias("hcu"), F.col("lpc.customer_hk") == F.col("hcu.customer_hk"), "left")
    .join(latest_customer_sat.alias("cus"), F.col("lpc.customer_hk") == F.col("cus.customer_hk"), "left")
    .join(payment_agg.alias("pa"), "claim_hk", "left")
    .select(
        F.col("hc.claim_number"),
        F.col("cs.claim_status"),
        F.col("cs.incident_date"),
        F.col("cs.reported_date"),
        F.col("cs.loss_type"),
        F.col("cs.claim_amount"),
        F.col("cs.reserve_amount"),
        F.col("cs.fraud_indicator"),
        F.col("hp.policy_number"),
        F.col("ps.coverage_type"),
        F.col("ps.deductible_amount"),
        F.col("ps.underwriting_status"),
        F.col("hcu.customer_id"),
        F.col("cus.full_name"),
        F.col("cus.risk_segment"),
        F.col("hv.vin"),
        F.col("vs.vehicle_make"),
        F.col("vs.vehicle_model"),
        F.col("vs.vehicle_year"),
        F.col("vs.garaging_state"),
        F.col("pa.payment_count"),
        F.col("pa.total_payment_amount"),
        F.col("pa.latest_payment_date")
    )
)

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 6, Finished, Available, Finished, False)

In [6]:
dm_claim_history_timeline = (
    hub_claim.alias("hc")
    .join(sat_claim_details.alias("scd"), "claim_hk", "inner")
    .select(
        F.col("hc.claim_number"),
        F.col("scd.effective_from"),
        F.col("scd.claim_status"),
        F.col("scd.loss_type"),
        F.col("scd.claim_amount"),
        F.col("scd.reserve_amount"),
        F.col("scd.fraud_indicator"),
        F.col("scd.record_source")
    )
    .orderBy("claim_number", "effective_from")
)

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 7, Finished, Available, Finished, False)

In [7]:
repeat_claimant = (
    silver_claims.groupBy("claimant_customer_id")
    .agg(F.countDistinct("claim_number").alias("claim_count_per_customer"))
)

dm_fraud_support_case = (
    dm_claim_360.alias("c360")
    .join(
        silver_claims.select("claim_number", "claimant_customer_id", "days_to_report").alias("sc"),
        "claim_number",
        "left"
    )
    .join(
        repeat_claimant.alias("rc"),
        F.col("sc.claimant_customer_id") == F.col("rc.claimant_customer_id"),
        "left"
    )
    .withColumn(
        "vehicle_policy_state_mismatch_flag",
        F.when(F.col("garaging_state").isNotNull() & F.col("garaging_state").isNotNull(), F.lit(0)).otherwise(F.lit(0))
    )
    .withColumn(
        "repeat_claimant_flag",
        F.when(F.col("claim_count_per_customer") > 1, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "open_claim_flag",
        F.when(F.col("claim_status").isin("Open", "Investigating"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "high_claim_amount_flag",
        F.when(F.col("claim_amount") >= 10000, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "multiple_payments_flag",
        F.when(F.col("payment_count") > 1, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "suspected_fraud_flag",
        F.when(
            (F.col("fraud_indicator") == "Y") |
            (F.col("repeat_claimant_flag") == 1) |
            (F.col("multiple_payments_flag") == 1) |
            (F.col("high_claim_amount_flag") == 1),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .select(
        "claim_number",
        F.col("sc.claimant_customer_id").alias("claimant_customer_id"),
        "claim_status",
        "loss_type",
        "claim_amount",
        "reserve_amount",
        "days_to_report",
        "payment_count",
        "total_payment_amount",
        "risk_segment",
        "repeat_claimant_flag",
        "open_claim_flag",
        "high_claim_amount_flag",
        "multiple_payments_flag",
        "suspected_fraud_flag"
    )
)

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 8, Finished, Available, Finished, False)

In [8]:
dm_claim_360.write.mode("overwrite").format("delta").saveAsTable("dm_claim_360")
dm_claim_history_timeline.write.mode("overwrite").format("delta").saveAsTable("dm_claim_history_timeline")
dm_fraud_support_case.write.mode("overwrite").format("delta").saveAsTable("dm_fraud_support_case")

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 9, Finished, Available, Finished, False)

In [9]:
for table_name in [
    "dm_claim_360",
    "dm_claim_history_timeline",
    "dm_fraud_support_case"
]:
    print(f"\n=== {table_name} ===")
    spark.sql(f"SELECT COUNT(*) AS row_count FROM {table_name}").show()

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 10, Finished, Available, Finished, False)


=== dm_claim_360 ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== dm_claim_history_timeline ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== dm_fraud_support_case ===
+---------+
|row_count|
+---------+
|       12|
+---------+



In [10]:
spark.table("dm_claim_360").show(truncate=False)
spark.table("dm_claim_history_timeline").show(truncate=False)
spark.table("dm_fraud_support_case").show(truncate=False)

StatementMeta(, f8acc74a-2132-41ab-97d2-4c453efdda93, 11, Finished, Available, Finished, False)

+------------+-------------+-------------+-------------+---------+------------+--------------+---------------+-------------+-------------+-----------------+-------------------+-----------+-------------+------------+--------+------------+-------------+------------+--------------+-------------+--------------------+-------------------+
|claim_number|claim_status |incident_date|reported_date|loss_type|claim_amount|reserve_amount|fraud_indicator|policy_number|coverage_type|deductible_amount|underwriting_status|customer_id|full_name    |risk_segment|vin     |vehicle_make|vehicle_model|vehicle_year|garaging_state|payment_count|total_payment_amount|latest_payment_date|
+------------+-------------+-------------+-------------+---------+------------+--------------+---------------+-------------+-------------+-----------------+-------------------+-----------+-------------+------------+--------+------------+-------------+------------+--------------+-------------+--------------------+----------------